# Sta-RU Video Dubbing

End-to-end pipeline: YouTube URLs → dubbed videos with voice cloning.

**Workflow:**
1. Run all setup cells (GPU check, install, mount Drive).
2. Paste YouTube URLs (one per line) or upload a `.txt` / `.csv`.
3. Upload subtitle files named `{N#}-{LANG}.srt` (e.g. `1-EN.srt`).
4. Set options (target language, output folder, range, etc.).
5. Run the **Preview** cell — checks the first 3 URLs.
6. Run the **Batch** cell — processes everything end-to-end.

**Requires:** Colab runtime with GPU (Runtime → Change runtime type → T4 GPU).

## 1. GPU check

In [ ]:
!nvidia-smi | head -20

## 2. Install dependencies

First run takes ~5 min. Re-runs are instant.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg rubberband-cli

In [ ]:
!pip install -q yt-dlp coqui-tts pyrubberband srt soundfile numpy scipy
!pip install -q demucs           # vocal removal (optional)
!pip install -q googletrans==4.0.0rc1  # title translation (optional)
!pip install -q ipywidgets pandas tqdm

## 3. Mount Google Drive (optional)

Needed if you want outputs saved to your Drive. Skip if you'll download to your PC at the end.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clone the Sta-RU repo

We only need `colab/batch_dub.py` from here.

In [ ]:
import os, sys
REPO_DIR = '/content/Sta-RU'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/lazy-money/sta-ru.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull --quiet
sys.path.insert(0, os.path.join(REPO_DIR, 'colab'))
print('Repo ready at', REPO_DIR)

## 5. Provide YouTube URLs

Pick **one** of the methods below. N# is assigned in order: first URL becomes N#1, second N#2, and so on. This number is how subtitles get matched (`1-EN.srt` ↔ first URL).

**Method A — paste in the textarea**, **Method B — upload a `.txt`/`.csv`**.

In [ ]:
import ipywidgets as W
from IPython.display import display

url_box = W.Textarea(
    value='',
    placeholder='Paste one YouTube URL per line, e.g.\nhttps://www.youtube.com/watch?v=XzHwPDZAXnM\nhttps://www.youtube.com/watch?v=wh23XTOE5jU',
    description='URLs:',
    layout=W.Layout(width='100%', height='200px'),
)
display(url_box)

In [ ]:
# Or upload a .txt or .csv file with one URL per line/row
from google.colab import files

uploaded_urls_path = None
print('Click "Choose Files" and pick a .txt or .csv with your URLs (or skip this cell if you pasted above):')
_uploaded = files.upload()
if _uploaded:
    name = list(_uploaded.keys())[0]
    uploaded_urls_path = f'/content/{name}'
    with open(uploaded_urls_path, 'wb') as f:
        f.write(_uploaded[name])
    print(f'Loaded: {uploaded_urls_path}')
else:
    print('No file uploaded — will use the textarea.')

## 6. Upload subtitle files

Filenames must follow `{N#}-{LANG}.srt`, where `{N#}` matches the order of the URLs above and `{LANG}` is the 2-letter language code (uppercase).

Examples: `1-EN.srt`, `2-EN.srt`, `47-ES.srt`, `101-DE.srt`.

**Supported language codes:** `EN, ES, FR, DE, IT, PT, PL, TR, RU, NL, CS, AR, ZH, HU, KO, JA, HI`.

You can select multiple files at once.

In [ ]:
from google.colab import files
import os, re, shutil

SRT_DIR = '/content/srts'
os.makedirs(SRT_DIR, exist_ok=True)

print('Select your .srt files (you can multi-select):')
_srts = files.upload()
_pattern = re.compile(r'^(\d+)-([A-Z]{2})\.srt$')

loaded = []
bad = []
for name, content in _srts.items():
    target = os.path.join(SRT_DIR, name)
    with open(target, 'wb') as f:
        f.write(content)
    m = _pattern.match(name)
    if m:
        loaded.append((int(m.group(1)), m.group(2)))
    else:
        bad.append(name)

print(f'\nLoaded {len(loaded)} SRT files in {SRT_DIR}')
if loaded:
    by_lang = {}
    for n, lg in loaded:
        by_lang.setdefault(lg, []).append(n)
    for lg, ns in by_lang.items():
        print(f'  {lg}: N# {sorted(ns)}')
if bad:
    print(f'\n[WARN] Wrong filename format (ignored by N# matching): {bad}')

## 7. Options

Tweak the widgets, then run the next cell to lock in the configuration.

In [ ]:
import ipywidgets as W
from IPython.display import display

LANG_OPTIONS = ['EN', 'ES', 'FR', 'DE', 'IT', 'PT', 'PL', 'TR', 'RU', 'NL', 'CS', 'AR', 'ZH', 'HU', 'KO', 'JA', 'HI']

lang_w = W.Dropdown(options=LANG_OPTIONS, value='EN', description='Target language:')
range_w = W.Text(value='all', description='Range (N#):', placeholder="e.g. 'all', '1-10', '47', '1-5,12,20-25'")
remove_voice_w = W.Checkbox(value=True, description='Remove original voice (Demucs)')
translate_title_w = W.Checkbox(value=False, description='Translate title to target language (googletrans)')

out_mode_w = W.RadioButtons(
    options=[('Save to Google Drive', 'drive'), ('Save locally in Colab (download manually)', 'local')],
    value='drive',
    description='Output:',
    layout=W.Layout(width='600px'),
)
out_path_w = W.Text(value='/content/drive/MyDrive/Dubbing/EN', description='Output dir:', layout=W.Layout(width='600px'))

def _sync_default_out(change):
    lg = lang_w.value
    if out_mode_w.value == 'drive':
        out_path_w.value = f'/content/drive/MyDrive/Dubbing/{lg}'
    else:
        out_path_w.value = f'/content/output/{lg}'
lang_w.observe(_sync_default_out, names='value')
out_mode_w.observe(_sync_default_out, names='value')

display(lang_w, range_w, remove_voice_w, translate_title_w, out_mode_w, out_path_w)
print('Adjust the values above, then run the next cell to lock them in.')

In [ ]:
# Lock in configuration
CONFIG = {
    'lang':              lang_w.value,
    'range_expr':        range_w.value,
    'remove_voice':      remove_voice_w.value,
    'translate_titles':  translate_title_w.value,
    'output_dir':        out_path_w.value,
    'srt_dir':           SRT_DIR,
}

# Resolve URL source: textarea takes precedence, else uploaded file
url_text = (url_box.value or '').strip()
if url_text:
    URL_SOURCE = [l for l in url_text.splitlines() if l.strip()]
    print(f'URL source: textarea ({len(URL_SOURCE)} lines)')
elif uploaded_urls_path:
    URL_SOURCE = uploaded_urls_path
    print(f'URL source: {uploaded_urls_path}')
else:
    raise RuntimeError('No URLs provided. Paste in the textarea or upload a file.')

print('\nConfiguration:')
for k, v in CONFIG.items():
    print(f'  {k:18} = {v}')

## 8. Preview

Fetches metadata for up to the first 3 URLs (title, upload date, duration) so you can verify before running the full batch.

In [ ]:
import pandas as pd
from batch_dub import load_urls, build_items, build_output_name

preview_urls = load_urls(URL_SOURCE)[:3]
print(f'Previewing {len(preview_urls)} of {len(load_urls(URL_SOURCE))} URL(s)...')

preview_items = build_items(preview_urls, translate_titles=CONFIG['translate_titles'], target_lang=CONFIG['lang'].lower())

rows = []
for it in preview_items:
    rows.append({
        'N#': it.n,
        'Date': it.upload_date,
        'Duration': f'{int(it.duration//60)}:{int(it.duration%60):02d}' if it.duration else '',
        'Title': it.title or '(metadata failed)',
        'Title (translated)': it.title_translated if CONFIG['translate_titles'] else '—',
        'Output filename': build_output_name(it, CONFIG['lang'], CONFIG['translate_titles']) if it.title else '',
    })
pd.DataFrame(rows).style.set_properties(**{'text-align': 'left'})

## 9. Run batch

Processes every URL whose `N#` falls in the configured range **and** has a matching SRT uploaded.

Per-video timing depends on video length and GPU. Roughly 3-5× faster than realtime on a T4 (so a 10-min video takes 2-3 min).

In [ ]:
from batch_dub import run_batch

results = run_batch(
    urls=URL_SOURCE,
    srt_dir=CONFIG['srt_dir'],
    output_dir=CONFIG['output_dir'],
    lang=CONFIG['lang'],
    translate_titles=CONFIG['translate_titles'],
    remove_voice=CONFIG['remove_voice'],
    range_expr=CONFIG['range_expr'],
)

## 10. Download outputs (local mode only)

If you saved to Drive, you're done — go to your Drive folder. If you saved locally, run this to download:

In [ ]:
from google.colab import files
from pathlib import Path

out_dir = Path(CONFIG['output_dir'])
for it in results:
    if it.status == 'done' and it.output_path and it.output_path.exists() and 'drive' not in str(out_dir):
        print(f'Downloading {it.output_path.name}...')
        files.download(str(it.output_path))